# 🧭 GIADA roadmap Task 11 — operatore causale Ca-HVA
Un solo run per H0–H4, su Ca-HVA+pas a un compartimento. La fase 11a controlla ingressi e oracle; la fase 11b confronta integratori e modelli appresi. L'interfaccia esterna è 1 ms. Serve GPU CUDA per il training. Nessun futuro teacher è passato ai modelli in inferenza.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_roadmap_task11')
GIADA_REPO=WORK/'giada';TEACHER_REPO=WORK/'neuron_as_deep_net'
assert not WORK.exists(),f'Directory già presente: {WORK}. Avvia una sessione nuova per evitare risultati mescolati.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip()
assert (GIADA_REPO/'src/giada_teacher/roadmap_causal_operator.py').is_file(),'Revisione GIADA non contiene la Task 11.'
print({'revision':REVISION,'teacher_revision':'074c4666300a8ad246601dab179a97a6942f0f29'})


## 📁 Unico input esterno
Aggiungi agli Input Kaggle `giada_cahva_active_closed_loop_microcanary.zip` (Task 7b), oppure il relativo Dataset estratto. La cella verifica hash e 24 episodi prima di usare le tracce. Non servono i dataset HayFlow da gigabyte.


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher import ExtractedGateFormula
from src.giada_teacher.roadmap_causal_operator import (CausalOperatorConfig,verify_native_anchor,generate_role,flatten_role,numerical_matrix,missing_input_counterfactual,train_matrix,evaluate_frozen)
import torch
assert torch.cuda.is_available(),'La matrice appresa preregistrata richiede GPU CUDA.'
config=CausalOperatorConfig();config.validate()
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
INPUT_ROOT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK7B_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
 candidates+=list(INPUT_ROOT.rglob('giada_cahva_active_closed_loop_microcanary.zip'))
 candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'cahva-active' in str(p).lower() or 'cahva_active' in str(p).lower()]
 candidates += [p.parent for p in INPUT_ROOT.rglob('trajectories.npz') if 'cahva' in str(p.parent).lower() and (p.parent/'final_report.json').is_file()]
TASK7B_SOURCE=None;native_anchor=None
for candidate in candidates:
 if not candidate.exists():continue
 try:native_anchor=verify_native_anchor(candidate,formula);TASK7B_SOURCE=candidate.resolve();break
 except (ValueError,RuntimeError,FileNotFoundError,KeyError):continue
assert TASK7B_SOURCE is not None,'Task 7b esatta non trovata: aggiungi giada_cahva_active_closed_loop_microcanary.zip agli Input Kaggle o imposta GIADA_TASK7B_ARTIFACT.'
display({'source':str(TASK7B_SOURCE),'native_anchor':native_anchor,'gpu':torch.cuda.get_device_name(0)})


## 11a — 🔬 ingressi e riferimenti
Generiamo episodi disgiunti per seed. I 4 valori di corrente del prossimo millisecondo sono pianificati prima dell'update. Questa cella crea train e development; il sealed viene creato solo dopo il congelamento della selezione.


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_roadmap_task11_causal_operator')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
OUTPUT_DIR.mkdir(parents=True)
train_role=generate_role(11011,config.train_episodes,config)
development_role=generate_role(11029,config.development_episodes,config)
train_data=flatten_role(train_role);development_data=flatten_role(development_role)
native_contract={'native_anchor':native_anchor,'formula_sha256':formula.source_sha256,'train_episode_count':config.train_episodes,'development_episode_count':config.development_episodes,'train_schedule_sha256':hashlib.sha256(train_role['current_na'].tobytes()).hexdigest(),'development_schedule_sha256':hashlib.sha256(development_role['current_na'].tobytes()).hexdigest(),'external_step_ms':1,'reference_dt_ms':config.reference_dt_ms}
(OUTPUT_DIR/'prepare_contract.json').write_text(json.dumps(native_contract,indent=2))
display({'native_valid':native_anchor['valid'],'train_windows':len(train_data['initial']),'development_windows':len(development_data['initial']),'input_widths':{'full':train_data['x_full'].shape[1],'reduced':train_data['x_reduced'].shape[1]},'missing_input_counterfactual':missing_input_counterfactual(development_role)})


## 11b — 🧠 tre bracci appaiati sulla GPU
`effect_full` e `effect_reduced` controllano l'informazione H0 e l'effetto integrato H3; `path_full` controlla H2. Stessi minibatch e checkpoint 50/100/200/400. I seed si selezionano esclusivamente sul development. H1 e H4 sono bracci numerici senza training nello stesso esperimento. Il tracker stampa una riga per checkpoint, non grandi array.


In [ ]:
freeze=train_matrix(train_role,development_role,config,OUTPUT_DIR)
display({'selected':freeze['selected'],'freeze_sha256':freeze['freeze_sha256'],'selection_source':freeze['selection_source'],'sealed_test_accessed':freeze['sealed_test_accessed']})
assert not freeze['sealed_test_accessed']


## 🔒 Valutazione sealed una volta sola
Il seed sealed è specificato nella configurazione preregistrata e non è usato per scegliere modello o soglie. La cella valuta tutti i bracci, inclusi gli oracle diagnostici e il rollout ricorsivo. Una F1 evento senza eventi positivi nel teacher viene riportata come assente, non come successo o fallimento.


In [ ]:
sealed_role=generate_role(config.sealed_role_seed,config.sealed_episodes,config)
final=evaluate_frozen(freeze,sealed_role,config,OUTPUT_DIR,native_anchor,code_revision=REVISION)
display({'valid':final['valid'],'native_anchor':final['native_anchor'],'H0_counterfactual':final['H0_input_contract']['same_reduced_input_counterfactual'],'H1':{k:{'V':round(v['voltage_rmse_mv'],5),'m':round(v['m_rmse'],6),'seconds':round(v['wall_seconds'],4)} for k,v in final['H1_coupled_numerical_stages'].items()},'H2_oracle_path':final['H2_oracle_compact_path'],'H3_oracle_effect':final['H3_oracle_integrated_effect']})
display({'one_step':final['learned_one_step'],'recursive_16ms':final['learned_recursive_16ms'],'sealed_used_for_selection':final['sealed_used_for_selection']})
assert final['valid'] and not final['sealed_used_for_selection'] and not final['future_teacher_voltage_as_model_input']


## 📦 Scarica il risultato
ZIP piccolo con report e checkpoint. Usa il metodo Blob/base64 concordato. Non stampare matrici o array estesi nel notebook.


In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task11_causal_operator','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
